In [1]:
# Standard library
import logging
import os
import warnings
import pathlib as pl

# Third-party libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import shapely.wkb
import tifffile
import timm
import torch
from PIL import Image
from tqdm.notebook import tqdm
from torchvision import transforms

# Warning filters
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Download data

We provide data to run the tutorial. This data is a crop of the 5k Ovarian Cancer Xenium experiment (https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-ovarian-cancer). 

You can **download the processed tutorial data here**: https://zenodo.org/records/17594071. Once you have downloaded the data, please put it in the data/ folder of tutorials or change the paths accordingly to point to the downloaded data.

In [3]:
adata = sc.read_h5ad('data/tutadata_subset.h5ad')

What do we expect in the adata object for the method to work:
- adata.obsm['spatial']: this should contain the X and Y coordinates of the cell/nuclei centroid in the high-resolution pixel space of the associated WSI.
- adata.X: this should be the cell x gene matrix of raw counts (! this needs to be single-cell resolution data)
- (optional): adata.obs['celltype']: the annotated celltypes (here called 'major_celltype').

In [2]:
source_image_path = 'data/wsi_crop.ome.tif'

with tifffile.TiffFile(source_image_path) as tif:
    wsi = tif.series[0].asarray()

This method requires a paired H&E high-resolution image. It can also work without transcriptomic data and just with the H&E image and X and Y coordinates extracted through segmentation (see the tutorial on extracting embeddings from H&E only).

# Unimodal embeddings

The input to the multimodal AE are low-dimensional embeddings from frozen foundation models. Here we show how to embed the transcriptomic data using **scGPT or Nicheformer** and the image data using **UNI2 or Virchow2**. The output of this section should be paired matrices, where for each cell, the transcriptomic matrix contains scGPT/Nicheformer embeddings and the image matrix contains UNI/Virchow embeddings. These are currently the foundation models supported by the framework but are susceptible to evolve as new FMs are developed. If you have a strong preference for another FM you can open an issue on Github to request it.

This part of the tutorial assumes you have:

- (a) **Requested access to UNI2 weights** and downloaded the checkpoint locally. The weights are located here: https://huggingface.co/MahmoodLab/UNI2-h/tree/main OR
- (b) **Requested access to Virchow2 weights**. You can request access on Huggingface: https://huggingface.co/paige-ai/Virchow2/tree/main.


- (a') For scGPT, we use code from **the Github repo sc_foundation_evals** (https://github.com/microsoft/zero-shot-scfoundation?tab=readme-ov-file). There are instructions on how to use their code and **download the scGPT weights** in their README, which we followed (the weights are here https://figshare.com/articles/dataset/Data_used_for_demo_of_the_code_accompanying_the_i_Assessing_the_limits_of_zero-shot_foundation_models_in_single-cell_biology_i_paper_/24747228?file=43480497) OR
- (b') **Downloaded Nicheformer weights** by following the instructions in their Github: https://github.com/theislab/nicheformer.

*Note*: If you wish you can always use the instructions from the original scGPT repository to embed data (https://scgpt.readthedocs.io/en/latest/index.html), as long as you obtain a cell x embedding matrix to feed into the next step. 

⚠️ You might run into some issues with torchtext, an error like 

```
OSError: /ewsc/yatesjos/spatialfusion_env/lib/python3.10/site-packages/torchtext/lib/libtorchtext.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
```

If so, you migh have to downgrade torch, see https://github.com/pytorch/text/issues/2272

We recommend using CUDA 12.1 to play well with this code, see main README for installation details.

**You only need to have one RNA-based embedding and one H&E based embedding. You can pick which embeddings to run in the following.**

## 1. scGPT

In [29]:
PATH_SCGPT_WEIGHTS = pl.Path('/ewsc/yatesjos/Broad_SpatialFoundation/scGPT_model/')
PATH_UNI_WEIGHTS = pl.Path('/ewsc/yatesjos/Broad_SpatialFoundation/UNI/pytorch_model.bin')

✅ Modify the sys path to include the parent directory where sc_foundation_evals is located to be able to use the functions.

In [9]:
import sys
sys.path.append('../../../Broad_SpatialFoundation/')

from sc_foundation_evals import cell_embeddings, scgpt_forward, data, model_output
from sc_foundation_evals.helpers.custom_logging import log

log.setLevel(logging.INFO)

/ewsc/yatesjos/spatialfusion_env/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/ewsc/yatesjos/spatialfusion_env/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/ewsc/yatesjos/spatialfusion_env/lib/python3.10/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/ewsc/yatesjos/spatialfusion_env/lib/python3.10/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last release

In [34]:
def run_embed_scGPT(
    dataset_path: str,
    model_dir: str,  # path to the pre-trained model, 3 files are expected: model_weights (best_model.pt), model args (args.json), and model vocab (vocab.json)
    output_dir: str,  # output_dir is the path to which the results should be saved
    n_hvg: int,
    gene_col: str = "index",  # in which column in adata.obs are gene names stored? if they are in index, the index will be copied to a column with this name
    layer_key: str = "X",  # where the raw counts are stored?
    log_norm: bool = False,  # are the values log_norm already?
    seed: int = 42,
    max_seq_len: int = 1200,  # maximum sequence of the input is controlled by max_seq_len, here I'm using the pretrained default
    batch_size: int = 32,  # batch_size depends on available GPU memory; should be a multiple of 8
    input_bins: int = 51,
    model_run: str = "pretrained",
    num_workers: int = 0,  # if you can use multithreading specify num_workers
) -> None:


    # create the model
    scgpt_model = scgpt_forward.scGPT_instance(
        saved_model_path=model_dir,
        model_run=model_run,
        batch_size=batch_size,
        save_dir=output_dir,
        num_workers=num_workers,
        explicit_save_dir=True,
    )

    # create config
    scgpt_model.create_configs(seed=seed, max_seq_len=max_seq_len, n_bins=input_bins)

    scgpt_model.load_pretrained_model()

    # This is to keep the model running if the amount of genes is small
    input_data = data.InputData(adata_dataset_path=dataset_path)
    vocab_list = scgpt_model.vocab.get_stoi().keys()

    adata = input_data.adata
    genes_in_vocab = adata.var_names.intersection(vocab_list)
    if len(genes_in_vocab) / len(adata.var_names) < 0.5:
        log.warning("Fewer than 50% of genes are found in the model vocab — continuing anyway.")
    
    adata._inplace_subset_var(genes_in_vocab)
    input_data.adata = adata

    input_data.preprocess_data(
        gene_vocab=vocab_list,
        model_type="scGPT",
        gene_col=gene_col,
        data_is_raw=not log_norm,
        counts_layer=layer_key,
        n_bins=input_bins,
        n_hvg=n_hvg,
    )

    scgpt_model.tokenize_data(
        data=input_data, input_layer_key="X_binned", include_zero_genes=False
    )

    scgpt_model.extract_embeddings(data=input_data)

    pd.DataFrame(
        input_data.adata.obsm["X_scGPT"],
        index=input_data.adata.obs["cell_id"] if "cell_id" in input_data.adata.obs.columns else input_data.adata.obs.index
    ).to_parquet(pl.Path(output_dir) / "scGPT.parquet")


In [14]:
# change this to wherever you saved the data 
base_dir = pl.Path('data/')
sample_name = 'SUBSET_Xenium_Ovarian-5k'
adata_path = base_dir / "tutadata_subset.h5ad"

model_dir_GPT = PATH_SCGPT_WEIGHTS

In [36]:
print(f'Starting for {sample_name}')
run_embed_scGPT(
    dataset_path=str(adata_path),
    model_dir=str(model_dir_GPT),
    output_dir=str(base_dir),  
    n_hvg=1200,
    gene_col="index",
    layer_key="X",
    log_norm=False,
    seed=42,
    max_seq_len=1200,
    batch_size=16,
    input_bins=51,
    model_run="pretrained",
    num_workers=0,
)

INFO     | 2025-10-23 11:04:42 | Using device cuda
WARNING  | 2025-10-23 11:04:42 | Overriding pre-trained config['save_dir'] with data (was /scratch/ssd004/datasets/cellxgene/save/cellxgene_census_human-May23-08-36-2023)
WARNING  | 2025-10-23 11:04:42 | Overriding pre-trained config['max_seq_len'] with 1200 (was 1200)
INFO     | 2025-10-23 11:04:42 | Loading vocab from /ewsc/yatesjos/Broad_SpatialFoundation/scGPT_model/vocab.json


Starting for SUBSET_Xenium_Ovarian-5k


INFO     | 2025-10-23 11:04:42 | Loading model from /ewsc/yatesjos/Broad_SpatialFoundation/scGPT_model/best_model.pt
WARNING  | 2025-10-23 11:04:43 | Loading partial model params from /ewsc/yatesjos/Broad_SpatialFoundation/scGPT_model/best_model.pt
WARNING  | 2025-10-23 11:04:43 | Cannot load transformer_encoder.layers.0.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2025-10-23 11:04:43 | Cannot load transformer_encoder.layers.0.self_attn.in_proj_bias with shape torch.Size([1536])
WARNING  | 2025-10-23 11:04:43 | Cannot load transformer_encoder.layers.1.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2025-10-23 11:04:43 | Cannot load transformer_encoder.layers.1.self_attn.in_proj_bias with shape torch.Size([1536])
WARNING  | 2025-10-23 11:04:43 | Cannot load transformer_encoder.layers.2.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2025-10-23 11:04:43 | Cannot load transformer_encoder.layers.2.self_attn.in_proj_bia

scGPT - INFO - Filtering genes by counts ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Log1p transforming ...
scGPT - INFO - Subsetting highly variable genes ...
scGPT - WARNING - No batch_key is provided, will use all cells for HVG selection.
scGPT - INFO - Binning data ...


INFO     | 2025-10-23 11:04:50 | Tokenizing data
INFO     | 2025-10-23 11:04:53 | Preparing dataloader
INFO     | 2025-10-23 11:04:53 | Saving config to data
INFO     | 2025-10-23 11:04:53 | Extracting embeddings
INFO     | 2025-10-23 11:04:53 | Extracting embeddings for batch 1/2919
INFO     | 2025-10-23 11:05:19 | Extracting embeddings for batch 292/2919
INFO     | 2025-10-23 11:05:43 | Extracting embeddings for batch 583/2919
INFO     | 2025-10-23 11:06:07 | Extracting embeddings for batch 874/2919
INFO     | 2025-10-23 11:06:31 | Extracting embeddings for batch 1165/2919
INFO     | 2025-10-23 11:06:55 | Extracting embeddings for batch 1456/2919
INFO     | 2025-10-23 11:07:19 | Extracting embeddings for batch 1747/2919
INFO     | 2025-10-23 11:07:43 | Extracting embeddings for batch 2038/2919
INFO     | 2025-10-23 11:08:07 | Extracting embeddings for batch 2329/2919
INFO     | 2025-10-23 11:08:30 | Extracting embeddings for batch 2620/2919
INFO     | 2025-10-23 11:08:54 | Extracting

## 2. Nicheformer

In [4]:
import gc
import gzip
import logging
import pathlib as pl
from datetime import datetime

import anndata as ad
import nicheformer
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import tqdm

from torch.utils.data import DataLoader

# ============================================================
# Utilities
# ============================================================

def set_seed(seed=42):

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Build symbol -> Ensembl mapping
# ============================================================

def build_symbol_to_ensembl_map(gtf_path):

    logging.info("Parsing GTF")

    records = []

    with gzip.open(gtf_path, "rt") as f:

        for line in f:

            if line.startswith("#"):
                continue

            fields = line.strip().split("\t")

            if fields[2] != "gene":
                continue

            attrs = fields[8]

            attr_dict = {}

            for item in attrs.split(";"):

                item = item.strip()

                if item == "":
                    continue

                key, value = item.split(" ", 1)

                attr_dict[key] = value.strip('"')

            gene_id = attr_dict.get("gene_id")
            gene_name = attr_dict.get("gene_name")

            if gene_id and gene_name:

                gene_id = gene_id.split(".")[0]

                records.append(
                    (
                        gene_name.upper(),
                        gene_id,
                    )
                )

    mapping_df = pd.DataFrame(
        records,
        columns=["gene_symbol", "ensembl_id"],
    ).drop_duplicates()

    symbol_to_ens = mapping_df.set_index(
        "gene_symbol"
    )["ensembl_id"].to_dict()

    logging.info(
        f"Parsed {len(symbol_to_ens)} mappings"
    )

    return symbol_to_ens


# ============================================================
# Remove Xenium controls
# ============================================================

def remove_control_probes(adata):

    mask = ~adata.var_names.str.upper().str.startswith(
        (
            "BLANK_",
            "NEGCONTROLCODEWORD",
            "NEGCONTROLPROBE",
            "ANTISENSE_",
        )
    )

    return adata[:, mask].copy()


# ============================================================
# Map genes to Ensembl
# ============================================================

def map_genes_to_ensembl(
    adata,
    symbol_to_ens,
):

    adata.var_names = (
        adata.var_names
        .str.strip()
        .str.upper()
    )

    adata.var_names_make_unique()

    adata.var["ensembl_id"] = [
        symbol_to_ens.get(g, None)
        for g in adata.var_names
    ]

    mapped = adata.var["ensembl_id"].notnull().sum()

    logging.info(
        f"Mapped genes: {mapped}/{adata.n_vars}"
    )

    adata = adata[
        :,
        adata.var["ensembl_id"].notnull()
    ].copy()

    adata.var_names = (
        adata.var["ensembl_id"]
        .astype(str)
    )

    adata.var_names_make_unique()

    return adata


# ============================================================
# Align to vocab
# ============================================================

def align_to_vocab(
    adata,
    vocab,
):

    vocab = vocab[
        :,
        vocab.var_names.str.startswith("ENSG")
    ].copy()

    common_genes = vocab.var_names.intersection(
        adata.var_names
    )

    logging.info(
        f"Common genes: {len(common_genes)}"
    )

    adata = adata[:, common_genes].copy()

    ordered_genes = [
        g for g in vocab.var_names
        if g in adata.var_names
    ]

    adata = adata[:, ordered_genes].copy()

    return adata, ordered_genes, vocab


# ============================================================
# Align technology mean
# ============================================================

def align_technology_mean(
    ordered_genes,
    vocab,
    tech_mean_path,
):

    technology_mean_full = np.load(
        tech_mean_path
    )

    tech_mean_map = dict(
        zip(vocab.var_names, technology_mean_full)
    )

    technology_mean = np.array([
        tech_mean_map[g]
        for g in ordered_genes
    ])

    return technology_mean.astype(np.float32)


# ============================================================
# Add metadata
# ============================================================

def add_metadata(adata):

    adata.obs["modality"] = 4
    adata.obs["species"] = 5
    adata.obs["assay"] = 9

    if "nicheformer_split" not in adata.obs.columns:

        adata.obs["nicheformer_split"] = "train"

    return adata


# ============================================================
# Main embedding function
# ============================================================

def run_embed_nicheformer(
    dataset_path,
    output_dir,
    symbol_to_ens,
):

    logging.info(f"Loading {dataset_path}")

    adata = ad.read_h5ad(dataset_path)

    logging.info(f"Original shape: {adata.shape}")

    if "cell_id" in adata.obs:

        cell_ids = (
            adata.obs["cell_id"]
            .astype(str)
        )

    else:

        cell_ids = (
            adata.obs_names
            .astype(str)
        )

    # --------------------------------------------------------
    # Remove controls
    # --------------------------------------------------------

    adata = remove_control_probes(
        adata
    )

    logging.info(
        f"After removing controls: {adata.shape}"
    )

    # --------------------------------------------------------
    # Map to Ensembl
    # --------------------------------------------------------

    adata = map_genes_to_ensembl(
        adata,
        symbol_to_ens,
    )

    logging.info(
        f"After mapping: {adata.shape}"
    )

    # --------------------------------------------------------
    # Load vocab
    # --------------------------------------------------------

    vocab = sc.read_h5ad(
        VOCAB_PATH
    )

    # --------------------------------------------------------
    # Align genes
    # --------------------------------------------------------

    adata, ordered_genes, vocab = align_to_vocab(
        adata,
        vocab,
    )

    logging.info(
        f"Final aligned shape: {adata.shape}"
    )

    # --------------------------------------------------------
    # Align tech mean
    # --------------------------------------------------------

    technology_mean = align_technology_mean(
        ordered_genes,
        vocab,
        TECH_MEAN_PATH,
    )

    # --------------------------------------------------------
    # Convert counts to float32
    # --------------------------------------------------------

    adata.X = adata.X.astype(
        np.float32
    )

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    adata = add_metadata(
        adata
    )

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    dataset = nicheformer.data.NicheformerDataset(
        adata=adata,
        technology_mean=technology_mean,
        split="train",
        max_seq_len=CONFIG["max_seq_len"],
        aux_tokens=CONFIG["aux_tokens"],
        chunk_size=CONFIG["chunk_size"],
        metadata_fields={
            "obs": [
                "modality",
                "species",
                "assay",
            ]
        },
    )

    logging.info(
        f"Token shape: {dataset.tokens.shape}"
    )

    # --------------------------------------------------------
    # Dataloader
    # --------------------------------------------------------

    dataloader = DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    model = (
        nicheformer.models.Nicheformer
        .load_from_checkpoint(
            checkpoint_path=MODEL_PATH,
            strict=False,
        )
    )

    model.eval()

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = model.to(device)

    # --------------------------------------------------------
    # Generate embeddings
    # --------------------------------------------------------

    embeddings = []

    with torch.no_grad():

        for batch in tqdm.tqdm(dataloader):

            batch = {
                k: v.to(device)
                if isinstance(v, torch.Tensor)
                else v
                for k, v in batch.items()
            }

            emb = model.get_embeddings(
                batch=batch,
                layer=CONFIG["embedding_layer"],
            )

            embeddings.append(
                emb.cpu().numpy()
            )

            gc.collect()

    embeddings = np.concatenate(
        embeddings,
        axis=0,
    )

    # --------------------------------------------------------
    # Save parquet
    # --------------------------------------------------------

    outfile = (
        pl.Path(output_dir)
        / "nicheformer.parquet"
    )

    pd.DataFrame(
        embeddings,
        index=cell_ids,
    ).to_parquet(outfile)

    logging.info(
        f"Saved embeddings to {outfile}"
    )

/ewsc/yatesjos/graphmaeenv/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
2026-07-08 15:32:27,699 | INFO | Starting Nicheformer batch embedding


In [ ]:
# ============================================================
# Paths
# ============================================================

MODEL_PATH = (
    "../../../Broad_SpatialFoundation/"
    "nicheformer_model/nicheformer.ckpt"
)

VOCAB_PATH = (
    "../../../Broad_SpatialFoundation/"
    "nicheformer_model/model.h5ad"
)

TECH_MEAN_PATH = (
    "../../../Broad_SpatialFoundation/"
    "nicheformer_model/xenium_mean_script.npy"
)

GTF_PATH = (
    "../../../Broad_SpatialFoundation/"
    "gencode.v48.basic.annotation.gtf.gz"
)

base_dir = pl.Path('data/')
sample_name = 'SUBSET_Xenium_Ovarian-5k'
adata_path = base_dir / "tutadata_subset.h5ad"
output_dir = pl.Path("data/")

# ============================================================
# Config
# ============================================================

CONFIG = {
    "batch_size": 32,
    "max_seq_len": 1500,
    "aux_tokens": 30,
    "chunk_size": 1000,
    "num_workers": 0,
    "embedding_layer": -1,
}

# ============================================================
# Logging
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

log_file = BASE_DIR / f"nicheformer_batch_{timestamp}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(),
    ],
)

logging.info("Starting Nicheformer batch embedding")


In [ ]:
set_seed(42)

symbol_to_ens = (
    build_symbol_to_ensembl_map(
        GTF_PATH
    )
)

logging.info(
    f"Processing {sample_name}"
)

output_dir.mkdir(
    exist_ok=True
)

outfile = (
    output_dir
    / "nicheformer.parquet"
)

try:

    run_embed_nicheformer(
        dataset_path=adata_path,
        output_dir=output_dir,
        symbol_to_ens=symbol_to_ens,
    )

    logging.info(
        f"Finished {sample_name}"
    )

except Exception as e:

    logging.exception(
        f"FAILED {sample_name}: {e}"
    )

2026-07-08 15:33:55,051 | INFO | Parsing GTF
2026-07-08 15:33:58,806 | INFO | Parsed 77073 mappings
2026-07-08 15:33:58,812 | INFO | Processing SUBSET_Xenium_Ovarian-5k
2026-07-08 15:33:58,814 | INFO | Loading data/tutadata_subset.h5ad
2026-07-08 15:33:59,243 | INFO | Original shape: (46691, 5101)
2026-07-08 15:33:59,339 | INFO | After removing controls: (46691, 5101)
2026-07-08 15:33:59,346 | INFO | Mapped genes: 5056/5101
2026-07-08 15:33:59,414 | INFO | After mapping: (46691, 5056)
2026-07-08 15:33:59,606 | INFO | Common genes: 5027
2026-07-08 15:33:59,761 | INFO | Final aligned shape: (46691, 5027)
100%|███████████████████████████████████████████| 47/47 [00:24<00:00,  1.91it/s]
2026-07-08 15:34:24,549 | INFO | Token shape: (46691, 1500)
/ewsc/yatesjos/graphmaeenv/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/utils.py:49: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.0.7, which is newer than your current Lightning version: v1.9.5
  ra

## 3. UNI

In [33]:
def load_UNI_model(model_path: str, device: str = "cuda"):
    timm_kwargs = {
        'model_name': 'vit_giant_patch14_224',
        'img_size': 224,
        'patch_size': 14,
        'depth': 24,
        'num_heads': 24,
        'init_values': 1e-5,
        'embed_dim': 1536,
        'mlp_ratio': 2.66667 * 2,
        'num_classes': 0,
        'no_embed_class': True,
        'mlp_layer': timm.layers.SwiGLUPacked,
        'act_layer': torch.nn.SiLU,
        'reg_tokens': 8,
        'dynamic_img_size': True
    }

    model = timm.create_model(pretrained=False, **timm_kwargs)
    model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=True)
    model.eval().to(device)

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])
    return model, transform


def embed_UNI(wsi, adata, he_coords, output_dir, model_path, batch_size = 128, device = "cuda"):

    print('Load UNI model')
    model, transform = load_UNI_model(model_path, device)

    os.makedirs(output_dir, exist_ok=True)

    embeddings = []
    cell_ids = []
    
    batch_imgs = []
    batch_ids = []
    
    print(f"Embedding {len(he_coords)} image patches in batches of {batch_size}...")
    for cid, (x, y) in tqdm(zip(adata.obs_names, he_coords), total=len(adata)):
        x, y = int(x), int(y)
        x0, x1 = x - 128, x + 128
        y0, y1 = y - 128, y + 128
    
        pad_x0 = max(0, -x0)
        pad_x1 = max(0, x1 - wsi.shape[1])
        pad_y0 = max(0, -y0)
        pad_y1 = max(0, y1 - wsi.shape[0])
    
        patch = np.pad(
            wsi[max(0, y0):min(wsi.shape[0], y1), max(0, x0):min(wsi.shape[1], x1)],
            ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)),
            mode="constant"
        )
    
        if patch.shape[:2] != (256, 256):
            continue
    
        tensor_img = transform(Image.fromarray(patch))
        batch_imgs.append(tensor_img)
        batch_ids.append(cid)
    
        if len(batch_imgs) == batch_size:
            img_tensor = torch.stack(batch_imgs).to(device)
    
            with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16):
                batch_embs = model(img_tensor).to(torch.float16).cpu().numpy()
    
            embeddings.extend(batch_embs)
            cell_ids.extend(batch_ids)
            batch_imgs.clear()
            batch_ids.clear()
    
    # Final batch (if any)
    if batch_imgs:
        img_tensor = torch.stack(batch_imgs).to(device)
        with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16):
            batch_embs = model(img_tensor).to(torch.float16).cpu().numpy()
        embeddings.extend(batch_embs)
        cell_ids.extend(batch_ids)

    # Save embedding matrix
    df = pd.DataFrame(embeddings, index=cell_ids)
    df.to_parquet(f"{output_dir}/UNI.parquet")
    print(f"Saved {len(df)} embeddings to {output_dir}/UNI.parquet")    

In [34]:
output_dir = 'data/'
device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = PATH_UNI_WEIGHTS

he_coords = adata.obsm['spatial']

In [35]:
embed_UNI(wsi, adata, he_coords, output_dir, model_path, batch_size = 512, device = device)

Load UNI model
Embedding 46691 image patches in batches of 512...


100%|████████████████████████████████████| 46691/46691 [05:12<00:00, 149.44it/s]


Saved 46691 embeddings to data//UNI.parquet


## 4. Virchow

In [36]:
import os
import pathlib as pl
import logging
from datetime import datetime

import anndata as ad
import numpy as np
import pandas as pd
import tifffile
import timm
import torch

from PIL import Image
from tqdm import tqdm

from timm.layers import SwiGLUPacked
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# ============================================================
# Embedding function
# ============================================================

def load_wsi(path):
    try:
        logging.info("Trying tifffile.imread()")
        wsi = tifffile.imread(path)
        logging.info(f"Loaded via imread: shape={wsi.shape}")
        return wsi

    except Exception as e:
        logging.warning(
            f"tifffile.imread failed ({type(e).__name__}: {e}); "
            "falling back to TiffFile.pages[0]"
        )

        with tifffile.TiffFile(path) as tif:
            page = tif.pages[0]
            wsi = page.asarray()

        logging.info(f"Loaded via page[0]: shape={wsi.shape}")
        return wsi


def embed_virchow2(
    wsi,
    cell_names,
    he_coords,
    output_file,
    batch_size=128,
):

    embeddings = []
    cell_ids = []

    batch_imgs = []
    batch_ids = []

    logging.info(
        f"Embedding {len(he_coords)} image patches "
        f"in batches of {batch_size}"
    )

    for cid, (x, y) in tqdm(
        zip(cell_names, he_coords),
        total=len(cell_names)
    ):

        x, y = int(x), int(y)

        # ====================================================
        # Extract centered 256x256 patch
        # ====================================================

        x0, x1 = x - 128, x + 128
        y0, y1 = y - 128, y + 128

        pad_x0 = max(0, -x0)
        pad_x1 = max(0, x1 - wsi.shape[1])

        pad_y0 = max(0, -y0)
        pad_y1 = max(0, y1 - wsi.shape[0])

        patch = np.pad(
            wsi[
                max(0, y0):min(wsi.shape[0], y1),
                max(0, x0):min(wsi.shape[1], x1)
            ],
            ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)),
            mode="constant"
        )

        # Skip malformed patches
        if patch.shape[:2] != (256, 256):
            continue

        # Convert to PIL image
        patch_pil = Image.fromarray(patch)

        # Apply official transforms
        tensor_img = transform(patch_pil)

        batch_imgs.append(tensor_img)
        batch_ids.append(cid)

        # ====================================================
        # Run batch inference
        # ====================================================

        if len(batch_imgs) == batch_size:

            img_tensor = torch.stack(batch_imgs).to(device)

            with torch.inference_mode(), torch.autocast(
                device_type=device,
                dtype=torch.float16
            ):

                # Forward pass
                output = model(img_tensor)

                # Virchow2 token extraction
                class_token = output[:, 0]
                patch_tokens = output[:, 5:]

                # Final 2560-dim embedding
                batch_embs = torch.cat(
                    [
                        class_token,
                        patch_tokens.mean(1)
                    ],
                    dim=-1
                )

                batch_embs = batch_embs.cpu().numpy()

            embeddings.extend(batch_embs)
            cell_ids.extend(batch_ids)

            batch_imgs.clear()
            batch_ids.clear()

    # ========================================================
    # Final partial batch
    # ========================================================

    if batch_imgs:

        img_tensor = torch.stack(batch_imgs).to(device)

        with torch.inference_mode(), torch.autocast(
            device_type=device,
            dtype=torch.float16
        ):

            output = model(img_tensor)

            class_token = output[:, 0]
            patch_tokens = output[:, 5:]

            batch_embs = torch.cat(
                [
                    class_token,
                    patch_tokens.mean(1)
                ],
                dim=-1
            )

            batch_embs = batch_embs.cpu().numpy()

        embeddings.extend(batch_embs)
        cell_ids.extend(batch_ids)

    # ========================================================
    # Save embeddings
    # ========================================================

    df = pd.DataFrame(
        embeddings,
        index=cell_ids,
    )

    df.to_parquet(output_file)

    logging.info(
        f"Saved {len(df)} embeddings to {output_file}"
    )

In [37]:
# Folder containing WSI .tif files
wsi_path = pl.Path(
    "data/wsi_crop.ome.tif"
)
output_dir = pl.Path('data/')
xy_coords_col = 'spatial'

# change this to wherever you saved the data 
base_dir = pl.Path('data/')
sample_name = 'SUBSET_Xenium_Ovarian-5k'

outfile = output_dir / "Virchow2.parquet"

device = "cuda" if torch.cuda.is_available() else "cpu"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

log_file = WSI_DIR / f"virchow2_batch_{timestamp}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logging.info("Starting Virchow2 batch embedding job")
logging.info(f"Using device: {device}")
logging.info(f"Log file: {log_file}")

2026-07-08 14:58:16,381 | INFO | Starting Virchow2 batch embedding job
2026-07-08 14:58:16,382 | INFO | Using device: cuda
2026-07-08 14:58:16,383 | INFO | Log file: data/virchow2_batch_20260708_145816.log


In [38]:
# ============================================================
# Load Virchow2 model ONCE
# ============================================================

logging.info("Loading Virchow2 model...")

model = timm.create_model(
    "hf-hub:paige-ai/Virchow2",
    pretrained=True,
    mlp_layer=SwiGLUPacked,
    act_layer=torch.nn.SiLU,
)

model.eval().to(device)

# Official Virchow2 transforms
transform = create_transform(
    **resolve_data_config(model.pretrained_cfg, model=model)
)

logging.info("Virchow2 model loaded")



2026-07-08 14:58:16,539 | INFO | Loading Virchow2 model...
2026-07-08 14:58:23,594 | INFO | Loading pretrained weights from Hugging Face hub (paige-ai/Virchow2)
2026-07-08 14:58:23,634 | INFO | [paige-ai/Virchow2] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-07-08 14:58:24,236 | INFO | Virchow2 model loaded


In [39]:
logging.info(f"Processing {sample_name}")

try:

    if "cell_id" in adata.obs:

        cell_names = (
            adata.obs["cell_id"]
            .astype(str)
            .values
        )

    else:

        cell_names = (
            adata.obs_names
            .astype(str)
        )

    he_coords = adata.obsm[xy_coords_col]

    logging.info(f"Loading WSI: {wsi_path}")

    wsi = load_wsi(wsi_path)

    logging.info(f"WSI shape: {wsi.shape}")

    embed_virchow2(
        wsi=wsi,
        cell_names=cell_names,
        he_coords=he_coords,
        output_file=outfile,
        batch_size=512,
    )

    logging.info(f"Finished {sample_name}")

except Exception as e:

    logging.exception(
        f"FAILED processing {sample_name}: {e}"
    )


2026-07-08 14:58:24,242 | INFO | Processing SUBSET_Xenium_Ovarian-5k
2026-07-08 14:58:24,247 | INFO | Loading WSI: data/wsi_crop.ome.tif
2026-07-08 14:58:24,247 | INFO | Trying tifffile.imread()
2026-07-08 14:58:24,453 | INFO | Loaded via imread: shape=(10000, 10000, 3)
2026-07-08 14:58:24,453 | INFO | WSI shape: (10000, 10000, 3)
2026-07-08 14:58:24,454 | INFO | Embedding 46691 image patches in batches of 512
100%|████████████████████████████████████| 46691/46691 [05:40<00:00, 137.17it/s]
2026-07-08 15:04:35,255 | INFO | Saved 46691 embeddings to data/Virchow2.parquet
2026-07-08 15:04:35,265 | INFO | Finished SUBSET_Xenium_Ovarian-5k
